# `JAXsim` Showcase: Parallel Simulation of a free-falling body

First, we install the necessary packages and import them.

In [ ]:
# @title Imports and setup
import os

# Deactivate GPU to avoid out of memory errors
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Install JAX and Gazebo

# Set environment variable to avoid GPU out of memory errors
%env XLA_PYTHON_CLIENT_MEM_PREALLOCATE=false

import time

import jax
import jax.numpy as jnp
import rod
from rod.builder.primitives import SphereBuilder, BoxBuilder

import jaxsim.typing as jtp
from jaxsim import logging
from jaxsim.api.common import VelRepr
from jaxsim.math import Adjoint
from jaxsim.api.model import JaxSimModel
from jaxsim.api.data import JaxSimModelData

from jaxsim.mujoco import (
    MujocoVideoRecorder,
    MujocoModelHelper,
    RodModelToMjcf,
    SdfToMjcf,
    UrdfToMjcf,
)
from jaxsim.mujoco.loaders import MujocoCamera

logging.set_logging_level(logging.LoggingLevel.INFO)
logging.info(f"Running on {jax.devices()}")

We will use a simple sphere model to simulate a free-falling body. The spheres set will be composed of 9 spheres, each with a different position. The spheres will be simulated in parallel, and the simulation will be run for 3000 steps corresponding to 3 seconds of simulation.

**Note**: Parallel simulations are independent of each other, the different position is imposed only to show the parallelization visually.

In [ ]:
# rod_model = rod.Sdf(
#     version="1.7",
#     model=rod.Model(
#         name="single_pendulum",
#         link=[
#             rod.Link(
#                 name="base_link",
#                 inertial=rod.Inertial(mass=100.0, inertia=rod.Inertia()),
#                 collision=rod.Collision(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 2.15])),
#                     pose=rod.Pose(pose=[0, 0, 1, 0, 0, 0]),
#                     name="base_link_fixed_joint_lump__base_collision",
#                 ),
#                 visual=rod.Visual(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 2.15])),
#                     pose=rod.Pose(pose=[0, 0, 1, 0, 0, 0]),
#                     name="base_link_fixed_joint_lump__base_visual",
#                 ),
#             ),
#             rod.Link(
#                 name="upper_link",
#                 pose=rod.Pose(pose=[0, 0, 0, 0, 0, 0], relative_to="upper_joint"),
#                 inertial=rod.Inertial(
#                     mass=0.5,
#                     inertia=rod.Inertia(),
#                     pose=rod.Pose([0, 0, 0.5, 0, 0, 0]),
#                 ),
#                 collision=rod.Collision(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 1.0])),
#                     pose=rod.Pose(pose=[0, 0, 0.5, 0, 0, 0]),
#                     name="upper_link_collision",
#                 ),
#                 visual=rod.Visual(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 1.0])),
#                     pose=rod.Pose(pose=[0, 0, 0.5, 0, 0, 0]),
#                     name="upper_link_visual",
#                 ),
#             ),
#         ],
#         joint=[
#             rod.Joint(
#                 name="upper_joint",
#                 type="revolute",
#                 parent="base_link",
#                 child="upper_link",
#                 axis=rod.Axis(
#                     xyz=rod.Xyz([1, 0, 0]),
#                     limit=rod.Limit(lower=-5, upper=5),
#                 ),
#                 pose=rod.Pose(
#                     pose=[0.15, 0, 2.0, -1.5708, 0, 0], relative_to="base_link"
#                 ),
#             ),
#         ],
#     ),
# )

# model_sdf_string = rod.urdf.exporter.UrdfExporter.sdf_to_urdf_string(
#     sdf=rod_model.models()[0]
# )

In [ ]:
# @title Create a sphere model
model_sdf_string = rod.Sdf(
    version="1.7",
    model=BoxBuilder(x=0.30, y=0.30, z=1.0, mass=1.0, name="box")
    # model=SphereBuilder(radius=0.15, mass=1.0, name="sphere")
    .build_model().add_link().add_inertial().add_visual().add_collision().build(),
).serialize(pretty=True)

# import urllib

# url = "https://raw.githubusercontent.com/icub-tech-iit/ergocub-gazebo-simulations/master/models/stickBot/model.urdf"

# model_sdf_string = urllib.request.urlopen(url).read().decode()
# model_sdf_string = pathlib.Path("/home/flferretti/git/element_rl-for-codesign/assets/model/hopper.sdf")

JAXsim offers a simple functional API in order to interact in a memory-efficient way with the simulation. Four main elements are used to define a simulation:

- `model`: an object that defines the dynamics of the system.
- `data`: an object that contains the state of the system.
- `integrator`: an object that defines the integration method.
- `integrator_state`: an object that contains the state of the integrator.

In [ ]:
import jaxsim.api as js
from jaxsim import integrators
import jaxopt
from jaxsim.terrain import PlaneTerrain

dt = 0.001
integration_time = int(3.0 / dt)

plane_normal = (0.0, -0.4, 0.5)

model = js.model.JaxSimModel.build_from_model_description(
    model_description=model_sdf_string,
    is_urdf=True,
    terrain=PlaneTerrain(plane_normal=plane_normal),
)

# model = js.model.reduce(
#     model=model,
#     considered_joints=tuple(
#         [
#             j
#             for j in model.joint_names()
#             if "camera" not in j
#             # and "neck" not in j
#             # and "wrist" not in j
#             # and "thumb" not in j
#             # and "index" not in j
#             # and "middle" not in j
#             # and "ring" not in j
#             # and "pinkie" not in j
#             # and "elbow" not in j
#             # and "shoulder" not in j
#             # and "hip" not in j
#             # and "knee" not in j
#             and "lidar" not in j and "torso" not in j
#         ]
#     ),
# )
# model = js.model.reduce(
#     model=model,
#     considered_joints=(
#         "l_hip_pitch",  # 0
#         "l_shoulder_pitch",  # 1
#         "r_hip_pitch",  # 2
#         "r_shoulder_pitch",  # 3
#         "l_hip_roll",  # 4
#         "l_shoulder_roll",  # 5
#         "r_hip_roll",  # 6
#         "r_shoulder_roll",  # 7
#         "l_hip_yaw",  # 8
#         "l_shoulder_yaw",  # 9
#         "r_hip_yaw",  # 10
#         "r_shoulder_yaw",  # 11
#         "l_knee",  # 12
#         "l_elbow",  # 13
#         "r_knee",  # 14
#         "r_elbow",  # 15
#         "l_ankle_pitch",  # 16
#         "r_ankle_pitch",  # 17
#         "l_ankle_roll",  # 18
#         "r_ankle_roll",  # 19
#     ),
# )

# motdel = js.model.reduce(model=model, considered_joints=tuple())
from jaxsim.rbda.contacts.soft import SoftContactsParams

data = js.data.JaxSimModelData.build(
    model=model,
    contacts_params=SoftContactsParams.build(0.0, 0.0, 0.0),
)
integrator = integrators.fixed_step.RungeKutta4SO3.build(
    dynamics=js.ode.wrap_system_dynamics_for_integration(
        model=model,
        data=data,
        system_dynamics=js.ode.system_dynamics,
    ),
)

integrator_state = integrator.init(x0=data.state, t0=0.0, dt=dt)

In [ ]:
mcjf_string, assets = UrdfToMjcf.convert(
    urdf=model_sdf_string,
    cameras=MujocoCamera.build_from_target_view(
        camera_name="recorder",
        lookat=(0, 0, 1.0),
        distance=5.0,
        azimut=180 - 45,
        elevation=-20,
        fovy=45,
    ),
    plane_normal=plane_normal,
)
mj_helper = MujocoModelHelper.build_from_xml(
    mjcf_description=mcjf_string, assets=assets
)
recorder = MujocoVideoRecorder(
    model=mj_helper.model, data=mj_helper.data, fps=int(1 / dt), width=640, height=480
)

It is possible to automatically choose a good set of parameters for the terrain. 

By default, in JaxSim a sphere primitive has 250 collision points. This can be modified by setting the `JAXSIM_COLLISION_SPHERE_POINTS` environment variable.

Given that at its steady-state the sphere will act on two or three points, we can estimate the ground parameters by explicitly setting the number of active points to these values.

In [ ]:
# import jaxlie
import numpy as np
import jaxlie

data = data.reset_base_quaternion(
    base_quaternion=jaxlie.SO3.from_rpy_radians(
        jnp.pi / 12, 0.0, 0.0
    ).as_quaternion_xyzw()[jnp.array([3, 0, 1, 2])]
)
# data = data.reset_base_velocity(
#     base_velocity=jnp.array([0.9, -0.4, 1.0, 0.5, 0.6, 0.0])
# )

Let's create a position vector for a 3x3 grid. Every sphere will be placed at a different height.

In [ ]:
# Primary Calculations
envs_per_row = 1  # @slider(2, 10, 1)
initial_height = 2.0
env_spacing = 0.5
edge_len = env_spacing * (2 * envs_per_row - 1)


# Create Grid
def grid(edge_len, envs_per_row):
    edge = jnp.linspace(-edge_len, edge_len, envs_per_row)
    xx, yy = jnp.meshgrid(edge, edge)

    poses = [
        [[1, 1, initial_height], [0, 0, 0]]
        for i in range(xx.shape[0])
        for j in range(yy.shape[0])
    ]

    return jnp.array(poses)


logging.info(f"Simulating {envs_per_row**2} environments")
poses = grid(edge_len, envs_per_row)

In order to parallelize the simulation, we first need to define a function `simulate` for a single element of the batch.

In [ ]:
active_collidable_points = (
    jnp.zeros_like(jnp.array(model.kin_dyn_parameters.contact_parameters.body))
    .astype(bool)
    # .at[0:4]
    # .at[0]
    # .at[1]
    # .at[0:2]
    .at[0:8]
    .set(True)
)

# Get the number of total and active collidable points.
num_of_collidable_points = int(active_collidable_points.size)
num_of_active_collidable_points = int(active_collidable_points.astype(int).sum())

In [ ]:
# Define a function to simulate a single model instance
def simulate(
    data: js.data.JaxSimModelData, integrator_state: dict, pose: jnp.array
) -> tuple:

    @jax.jit
    def _convert_mixed_to_inertial_forces(
        CW_F: jax.Array, data: js.data.JaxSimModelData
    ) -> jax.Array:

        W_H_C = js.contact.transforms(model=model, data=data)[
            active_collidable_points, :, :
        ]

        def mixed_to_inertial(W_H_C: jax.Array, CW_fl: jax.Array) -> jax.Array:
            W_Xf_CW = Adjoint.from_transform(
                W_H_C.at[0:3, 0:3].set(jnp.eye(3)),
                inverse=True,
            ).T
            return W_Xf_CW @ jnp.hstack([CW_fl, jnp.zeros(3)])

        W_f_active_C = jax.vmap(mixed_to_inertial)(W_H_C, CW_F)

        W_f_C = (
            jnp.zeros(shape=(num_of_collidable_points, 6))
            .at[active_collidable_points, :]
            .set(W_f_active_C)
        )

        parent_link_index_of_collidable_points = jnp.array(
            model.kin_dyn_parameters.contact_parameters.body
        )

        mask = parent_link_index_of_collidable_points[:, jnp.newaxis] == jnp.arange(
            model.number_of_links()
        )

        W_f_L = mask.T @ W_f_C

        return W_f_L

    @jax.jit
    def _linear_acceleration_of_collidable_points(
        model: js.model.JaxSimModel,
        data: js.data.JaxSimModelData,
        collidable_point_indices: jax.Array,
        references: js.references.JaxSimModelReferences | None = None,
    ) -> jax.Array:
        """"""

        references = (
            references
            if references is not None
            else js.references.JaxSimModelReferences.zero(model=model)
        )

        with (
            data.switch_velocity_representation(VelRepr.Body),
            references.switch_velocity_representation(data.velocity_representation),
        ):

            W_v̇_WB, s̈, *_ = js.ode.system_velocity_dynamics(
                model=model,
                data=data,
                joint_forces=references.joint_force_references(model=model),
                link_forces=references.link_forces(model=model, data=data),
            )
            W_ν̇ = jnp.hstack([W_v̇_WB, s̈])

        with data.switch_velocity_representation(VelRepr.Inertial):
            O_J_WL_W = js.contact.jacobian(
                model=model,
                data=data,
                output_vel_repr=VelRepr.Mixed,
            )

        with data.switch_velocity_representation(VelRepr.Inertial):
            W_ν = data.generalized_velocity()
            O_J̇_WL_W = js.contact.jacobian_derivative(
                model=model,
                data=data,
                output_vel_repr=VelRepr.Mixed,
            )

        O_v̇_WC = jax.vmap(
            lambda idx: O_J̇_WL_W[idx, :, :] @ W_ν + O_J_WL_W[idx, :, :] @ W_ν̇
        )(collidable_point_indices)

        return O_v̇_WC[:, 0:3]

    def _imp_aref(
        position: jtp.Array,
        velocity: jtp.Array,
        stiffness: jtp.Float,
        damping: jtp.Float,
    ) -> tuple[jtp.Array, jtp.Array]:
        """Calculates impedance and offset acceleration in constraint frame.

        Args:
            position: position in constraint frame
            velocity: velocity in constraint frame

        Returns:
            impedance: constraint impedance
            a_ref: offset acceleration in constraint frame
        """

        imp_x = jnp.abs(position) / width
        imp_a = (1.0 / jnp.power(mid, p - 1)) * jnp.power(imp_x, p)

        imp_b = 1 - (1.0 / jnp.power(1 - mid, p - 1)) * jnp.power(1 - imp_x, p)

        imp_y = jnp.where(imp_x < mid, imp_a, imp_b)

        imp = jnp.clip(ξ_min + imp_y * (ξ_max - ξ_min), ξ_min, ξ_max)
        imp = jnp.atleast_1d(jnp.where(imp_x > 1.0, ξ_max, imp))

        K = 1 / (ξ_max * timeconst * ζ) ** 2
        D = 2 / (ξ_max * timeconst)

        # When passing negative values, K and D represent a spring and damper, respectively.
        K = jnp.where(
            stiffness < 0, -stiffness / ξ_max**2, 1 / (ξ_max * timeconst * ζ) ** 2
        )
        D = jnp.where(damping < 0, -damping / ξ_max, 2 / (ξ_max * timeconst))

        a_ref = -jnp.atleast_1d(D * velocity + K * imp * position)

        return imp, a_ref

    @jax.jit
    def _delassus_matrix(
        model: js.model.JaxSimModel,
        data: js.data.JaxSimModelData,
        position: jax.Array,
    ) -> jax.Array:
        """"""

        M = js.model.free_floating_mass_matrix(model=model, data=data)

        sl = jnp.s_[active_collidable_points, 0:3, :]
        J_WC = jnp.vstack(
            jax.vmap(lambda j, height: j * (height < 0))(
                js.contact.jacobian(model=model, data=data)[sl], position[:, 2]
            )
        )

        return J_WC @ jnp.linalg.lstsq(M, J_WC.T)[0]

    @jax.jit
    def _regularizers(
        model: JaxSimModel,
        data: JaxSimModelData,
        position: jtp.Array,
        velocity: jtp.Array,
        stiffness: jtp.Float,
        damping: jtp.Float,
    ) -> tuple:
        """Compute the contact jacobian and the reference acceleration.

        Args:
            model (JaxSimModel): The jaxsim model.
            position (jtp.Vector): The position of the collidable point.

        Returns:
            tuple: A tuple containing the contact jacobian, the reference acceleration, and the contact radius.
        """

        def _compute_row(
            stiffness: jtp.Float,
            damping: jtp.Float,
            *,
            link_idx: jtp.Float,
            position: jtp.Array,
            velocity: jtp.Array,
        ) -> tuple[jtp.Array, jtp.Array]:

            # Compute the reference acceleration.
            imp, a_ref = _imp_aref(
                position=position,
                velocity=velocity,
                stiffness=stiffness,
                damping=damping,
            )

            # Compute the inverse inertia of the parent link.
            # I_inv = jnp.linalg.inv(link_inertia[link_idx][:3, :3])
            I = js.kin_dyn_parameters.LinkParameters.inertia_tensor(
                jax.tree_util.tree_map(
                    lambda l: l[link_idx], model.kin_dyn_parameters.link_parameters
                )
            )

            # Compute the regularization terms.
            R = (
                (2 * friction**2 * (1 - imp) / (imp + 1e-12))
                * (1 + friction**2)
                @ jnp.linalg.inv(I)
            )

            return jax.tree.map(lambda x: x * (position[2] < 0), (a_ref, R))

        a_ref, R = jax.tree.map(
            jnp.concatenate,
            (
                *jax.vmap(_compute_row, in_axes=(None, None))(
                    stiffness,
                    damping,
                    link_idx=jnp.array(
                        model.kin_dyn_parameters.contact_parameters.body
                    ),
                    position=position,
                    velocity=velocity,
                ),
            ),
        )
        return a_ref, jnp.diag(R)

    def compute_force(model, data, position, velocity, stiffness, damping):

        def _detect_contact(x: jtp.Array, y: jtp.Array, z: jtp.Array) -> jtp.Array:
            x, y, z = (jnp.squeeze(i) for i in (x, y, z))

            n̂ = model.terrain.normal(x=x, y=y).squeeze()
            h = jnp.array([0, 0, z - model.terrain.height(x=x, y=y)])

            return jnp.array([0, 0, jnp.dot(h, n̂)], dtype=float)

        position = jax.vmap(_detect_contact)(*position.T)

        with data.switch_velocity_representation(VelRepr.Mixed):
            a_ref, R = _regularizers(
                model=model,
                data=data,
                position=position,
                velocity=velocity,
                stiffness=stiffness,
                damping=damping,
            )

            G = _delassus_matrix(model=model, data=data, position=position)
            CW_al_free_WC = _linear_acceleration_of_collidable_points(
                model=model,
                data=data,
                collidable_point_indices=jnp.arange(num_of_collidable_points),
                references=None,
            )

            CW_al_free_WC_active = jax.vmap(lambda x, height: height < 0 * x)(
                CW_al_free_WC, position
            ).flatten()

        # Calculate quantities for the linear optimization problem.
        A = G + R
        b = CW_al_free_WC_active - a_ref

        # objective = lambda x: jnp.sum(0.5 * (A @ x + b) ** 2)
        objective = lambda x: jnp.sum(jnp.square(A @ x + b))

        # Compute the 3D linear force in C[W] frame
        opt = jaxopt.LBFGS(
            fun=objective,
            maxiter=100,
            tol=1e-10,
            maxls=100,
            history_size=10,
            max_stepsize=100.0,
            stop_if_linesearch_fails=True,
            # verbose=2,
        )

        CW_f_Ci = opt.run(init_params=jnp.zeros_like(b)).params.reshape(-1, 3)

        return CW_f_Ci

    data = data.reset_base_position(base_position=pose)
    x_t_i = []
    joint_positions = []
    forces = []
    link_positions = []
    base_orientations = []
    in_contact = []
    Jd_nu = []

    # l_foot = model.link_names().index("l_ankle_2")
    # r_foot = model.link_names().index("r_ankle_2")
    base = model.link_names().index("box_link")
    # base = model.link_names().index("base_link")
    # base = model.link_names().index("root_link")

    # Check https://mujoco.readthedocs.io/en/latest/modeling.html#solver-parameters
    timeconst, ζ, ξ_min, ξ_max, width, mid, p, friction = (
        0.01,
        1.0,
        0.9,
        0.95,
        0.0001,
        0.1,
        1.0,
        0.0,
    )

    stiffness, damping = jnp.zeros(shape=(1,)), jnp.zeros(shape=(1,))

    compute_force_jit = jax.jit(
        lambda data, position, velocity: compute_force(
            model=model,
            data=data,
            position=position,
            velocity=velocity,
            stiffness=stiffness,
            damping=damping,
        )
    )

    try:
        references = js.references.JaxSimModelReferences.build(
            model=model, velocity_representation=data.velocity_representation
        )

        for _ in range(integration_time):

            with (
                data.switch_velocity_representation(VelRepr.Mixed),
                references.switch_velocity_representation(data.velocity_representation),
            ):

                W_p_Ci, W_ṗ_Ci = js.contact.collidable_point_kinematics(
                    model=model, data=data
                )

                CW_F = compute_force_jit(data, W_p_Ci, W_ṗ_Ci)

            with references.switch_velocity_representation(VelRepr.Inertial):
                W_f_L = _convert_mixed_to_inertial_forces(CW_F=CW_F, data=data)

                references = references.apply_link_forces(
                    forces=W_f_L, model=model, data=data, additive=False
                )

            with (
                data.switch_velocity_representation(VelRepr.Inertial),
                references.switch_velocity_representation(data.velocity_representation),
            ):

                data, integrator_state = js.model.step(
                    dt=dt,
                    model=model,
                    data=data,
                    integrator=integrator,
                    integrator_state=integrator_state,
                    joint_forces=None,
                    link_forces=references.link_forces(model=model, data=data),
                )

                x_t_i.append(data.base_position())
                joint_positions.append(data.joint_positions())
                base_orientations.append(data.base_orientation())
                forces.append(references.link_forces(model=model, data=data).squeeze())
                link_positions.append(
                    js.link.transform(model=model, data=data, link_index=base)[3, :3]
                )
                in_contact.append(np.array(W_p_Ci)[:, 2] < 0)
                Jd_nu.append(
                    js.contact.jacobian_derivative(model, data)[:, :3]
                    @ data.generalized_velocity()
                )

                print(f"Simulated time: {_ * dt:.3f}s", end="\r")
        # finally:
        return (
            x_t_i,
            base_orientations,
            joint_positions,
            forces,
            link_positions,
            in_contact,
            Jd_nu,
        )
    except Exception as e:
        raise e

We will make use of `jax.vmap` to simulate multiple models in parallel. This is a very powerful feature of JAX that allows to write code that is very similar to the single-model case, but can be executed in parallel on multiple models.
In order to do so, we need to first apply `jax.vmap` to the `simulate` function, and then call the resulting function with the batch of different poses as input.

Note that in our case we are vectorizing over the `pose` argument of the function `simulate`, this correspond to the value assigned to the `in_axes` parameter of `jax.vmap`:

`in_axes=(None, None, 0)` means that the first two arguments of `simulate` are not vectorized, while the third argument is vectorized over the zero-th dimension.

In [ ]:
# Run and time the simulation
now = time.perf_counter()

# x_t = simulate_vectorized(data, integrator_state, poses[:, 0]).
x_t, base_orientations, joint_positions, forces, link_positions, in_contact, Jd_nu = (
    simulate(data, integrator_state, poses[:, 0])
)

comp_time = time.perf_counter() - now

logging.info(
    f"Running simulation with {envs_per_row**2} models took {comp_time} seconds."
)
logging.info(
    f"This corresponds to an RTF (Real Time Factor) of {(envs_per_row**2 * integration_time / comp_time):.2f}"
)

In [ ]:
print(f"{model.link_names()=}")

In [ ]:
(np.array(Jd_nu) == 0).all()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

in_contact = np.array(in_contact)

# plt.plot(
#     np.arange(len(in_contact[:])) * dt,
#     in_contact.any(axis=1)[:],
#     label="In contact",
# )
# plt.plot(
#     np.arange(len(x_t[:])) * dt,
#     np.array(Jd_nu).reshape(1500, 24)[:],
#     label=[f"JD_nu_{component}" for component in range(24)],
# )

plt.plot(np.arange(len(x_t[:100])) * dt, np.array(x_t)[:100, 2])
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("")
plt.title("W_p_L")
# plt.legend()
plt.show()

In [ ]:
from pathlib import Path

for pose, orientation, s_t in zip(x_t, base_orientations, joint_positions, strict=True):
    mj_helper.set_base_position(pose)
    mj_helper.set_base_orientation(orientation)
    # mj_helper.set_joint_positions(positions=s_t, joint_names=model.joint_names())
    recorder.record_frame()  # camera_name="recorder")

import datetime

import mediapy as media

media.show_video(recorder.frames, fps=1 / dt)

recorder.write_video(
    path=(
        save_path := Path.cwd()
        / Path(
            f"{model.name()}_{data.velocity_representation}_{datetime.datetime.now().isoformat()}.mp4"
        )
    ),
    exist_ok=True,
)

In [ ]:
save_path

Now let's extract the data from the simulation and plot it. We expect to see the height time series of each sphere starting from a different value.

In [ ]:
forces = np.array([force for force in forces])
link_positions = np.array(link_positions)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(
    np.arange(len(link_positions)) * dt,
    link_positions,
    # label=["L", "R"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Link CoM")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

in_contact = np.array(in_contact)

# plt.plot(
#     np.arange(len(in_contact[:])) * dt,
#     in_contact.any(axis=1)[:] * 1000,
#     label="In contact",
# )
plt.plot(
    np.arange(len(forces[:])) * dt,
    forces[:],
    label=["X", "Y", "Z", "Rx", "Ry", "Rz"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Force [N]")
plt.title("Contact forces")
plt.legend()
plt.show()